# 19 Exercises

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

**19-1.** The economics example in [Section 19.1](./19_1_sources_of_complementarity.ipynb) used a demand function that was linear in the
price. Construct a nonlinear demand function that has each of the characteristics described below.
Define a corresponding complementarity problem, using the data from [Figure 19-2](./19_1_sources_of_complementarity.ipynb#fig-19-2) as much as possible.
Use a solver such as PATH to compute an equilibrium solution. Compare this solution to those for
the constant-demand and linear-demand alternatives shown in [Section 19.1](./19_1_sources_of_complementarity.ipynb).

(a) For price `i` near zero the demand is near demzero[i] and is decreasing at a rate near
demrate[i]. After price `i` has increased substantially, however, both the demand and the rate
of decrease of the demand approach zero.

(b) For price `i` near zero the demand is approximately constant at demzero[i], but as price `i`
approaches demlim[i] the demand drops quickly to zero.

\(c) Demand for `i` actually rises with price, until it reaches a value demmax[i] at a price of
demarg[i]. Then demand falls with price.

**19-2.** For each scenario in the previous problem, experiment with different starting points for the
Level and Price values. Determine whether there appears a unique equilibrium point.

**19-3.** A bimatrix game between players A and B is defined by two m by n "payoff" matrices,
whose elements we denote by a `i` `j` and b `i` `j` . In one round of the game, player A has a choice of m
alternatives and player B a choice of n alternatives. If A plays (chooses) `i` and B plays j, then A
and B win amounts a `i` `j` and b `i` `j` , respectively; negative winnings are interpreted as losses.
We can allow for "mixed" strategies in which A plays `i` with probability `p` Ai and B plays `j` with
probability `p` Bj . Then the expected value of player A's winnings is:

```ampl
n
Σ a ij
j=1
          × p Bj , if A plays i
```

and the expected value of player B's winnings is:

```ampl
m
 Σ b ij
i =1
          × p Ai , if B plays j
```

A "pure" strategy is the special case in which each player has one probability equal to 1 and the
rest equal to 0.

A pair of strategies is said to represent a Nash equilibrium if neither player can improve his
expected payoff by changing only his own strategy.
(a) Show that the requirement for a Nash equilibrium is equivalent to the following
complementarity-like conditions:

```ampl
for all i such that p Ai > 0, A's expected return when playing i equals A'smaximum
expected return over all possible plays
  for all j such that p Bj > 0, B's expected return when playing j equals B's maximum
  expected return over all possible plays
```

(b) To build a complementarity problem in AMPL whose solution is a Nash equilibrium, the parameters
representing the payoff matrices can be defined by the following `param` declarations:

```ampl
param nA > 0;         # actions available to player A
param nB > 0;         # actions available to player B
param payoffA {1..nA, 1..nB};              # payoffs to player A
param payoffB {1..nA, 1..nB};              # payoffs to player B
```

The probabilities that define the mixed strategies are necessarily variables. In addition it is convenient
to define a variable to represent the maximum expected payoff for each player:

```ampl
var PlayA {i in 1..nA};            # player A's mixed strategy
var PlayB {j in 1..nB};            # player B's mixed strategy
var MaxExpA;         # maximum expected payoff to player A
var MaxExpB;         # maximum expected payoff to player B
```

Write AMPL declarations for the following constraints:

- The probabilities in any mixed strategy must be nonnegative.
- The probabilities in each player's mixed strategy must sum to 1.
- Player A's expected return when playing any particular `i`
must not exceed A's maximum expected return over all possible plays
- Player B's expected return when playing any particular `j`
must not exceed B's maximum expected return over all possible plays

\(c) Write an AMPL model for a square complementarity system that enforces the constraints in (b)
and the conditions in (a).

(d) Test your model by applying it to the "rock-scissors-paper" game in which both players have
the payoff matrix

```ampl
0    1    -1
-1    0     1
 1   -1     0
```

Confirm that an equilibrium is found where each player chooses between all three plays with equal
probability.

(e) Show that the game for which both players have the payoff matrix

```ampl
-3    1     3   -1
 2    3    -1   -5
```

has several equilibria, at least one of which uses mixed strategies and one of which uses pure
strategies.

Running a solver such as PATH will only return one equilibrium solution. To find more, experiment
with changing the initial solution or fixing some of the variables to 0 or 1.
**19-4.** Two companies have to decide now whether to adopt standard 1 or standard 2 for future
introduction in their products. If they decide on the same standard, company A has the greater payoff
because its technology is superior. If they decide on different standards, company B has the
greater payoff because its market share is greater. These considerations lead to a bimatrix game
whose payoff matrices are

```ampl
A = 10      3       B = 4      6
     2      9           7      5
```

(a) Use a solver such as PATH to find a Nash equilibrium. Verify that it is a mixed strategy, with
A's probabilities being 1/2 for both standards and B's probabilities being 3/7 and 4/7 for
standards 1 and 2, respectively.

Why is a mixed strategy not appropriate for this application?

(b) You can see what happens when company A decides on standard 1 by issuing the following
commands:

```ampl
ampl: fix PlayA[1] := 1;
ampl: solve;
presolve, constraint ComplA[1].L:
              all variables eliminated, but upper bound = -1 < 0
```

Explain how AMPL's presolve phase could deduce that the complementarity problem has no feasible
solution in this case.

\(c) Through further experimentation, show that there are no Nash equilibria for this situation that
involve only pure strategies.